## Load Data

In [71]:
import pickle
import os
import numpy as np
import pandas as pd 
from tqdm import tqdm

DATA_PATH = "D:/ML/RSNA2024"

with open(os.path.join(DATA_PATH, "featureExtraction.pkl"), "rb") as f:
    data = pickle.load(f)

In [17]:
features, labels, studyIds, seriesIds, instanceNrs = data["features"], data["labels"], data["studyIds"], data["seriesIds"], data["instanceNumbers"]

In [18]:
features = np.array(features)
labels = np.array(labels)
print(features.shape)
print(labels.shape)

(147218, 512)
(147218, 25)


In [19]:
from sklearn.model_selection import cross_val_score, train_test_split

# features = features[0:1000]
# labels = labels[0:1000]

X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.33, random_state=42)

## Decision Tree

In [20]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=0)


In [21]:
# clf.fit(X_train, y_train)
# clf.predict_proba(X_test, y_test)

## Random Forest

In [35]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

clfRF = RandomForestClassifier()
multi_target_rf = MultiOutputClassifier(clfRF, n_jobs=-1)

# Fit the model
multi_target_rf.fit(X_train, y_train)



Accuracy for label 1: 1.00
F1 for label 1: 1.00
Accuracy for label 2: 1.00
F1 for label 2: 1.00
Accuracy for label 3: 1.00
F1 for label 3: 1.00
Accuracy for label 4: 1.00
F1 for label 4: 1.00
Accuracy for label 5: 1.00
F1 for label 5: 1.00
Accuracy for label 6: 1.00
F1 for label 6: 1.00
Accuracy for label 7: 1.00
F1 for label 7: 1.00
Accuracy for label 8: 1.00
F1 for label 8: 1.00
Accuracy for label 9: 0.99
F1 for label 9: 0.99
Accuracy for label 10: 1.00
F1 for label 10: 1.00
Accuracy for label 11: 1.00
F1 for label 11: 1.00
Accuracy for label 12: 1.00
F1 for label 12: 1.00
Accuracy for label 13: 1.00
F1 for label 13: 1.00
Accuracy for label 14: 0.99
F1 for label 14: 0.99
Accuracy for label 15: 1.00
F1 for label 15: 1.00
Accuracy for label 16: 1.00
F1 for label 16: 1.00
Accuracy for label 17: 1.00
F1 for label 17: 1.00
Accuracy for label 18: 1.00
F1 for label 18: 1.00
Accuracy for label 19: 0.99
F1 for label 19: 0.99
Accuracy for label 20: 1.00
F1 for label 20: 1.00
Accuracy for label

In [24]:
# Predict on the test set
Y_pred = multi_target_rf.predict(X_test)

# Evaluate accuracy for each label separately
accuracies = []
f1s = []
for i in range(y_test.shape[1]):
    acc = accuracy_score(y_test[:, i], Y_pred[:, i])
    f1 = f1_score(y_test[:, i], Y_pred[:, i], average="micro")
    accuracies.append(acc)
    f1s.append(f1)
    print(f'Accuracy for label {i + 1}: {acc:.2f}')
    print(f'F1 for label {i + 1}: {f1:.2f}')

# Calculate and print the mean accuracy across all labels
mean_accuracy = np.mean(accuracies)
print(f'Mean accuracy across all labels: {mean_accuracy:.2f}')
mean_f1 = np.mean(f1s)
print(f'Mean F1 across all labels: {mean_f1:.2f}')


## Ideas


- Feature Selection von X ( shape = (147218, 512) )
  - Resultat sind N wichtige Features --> shape = (147218, N)
- Sortiere die Features nach studyId und ansteigender InstanceNr --> studyId : [transv1, transv2, ..., sagital1, sagital2, ..., ]
  - Resultat shape = (#StudyIds, VariableAnzahlAnBildern, N)
  - Einfachste Möglichkeit mean(axis=1) --> (#StudyIds, 1, N)
   

In [72]:
df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
df = df.fillna("Normal/Mild")
df.set_index("study_id", inplace=True)
allLabels = np.array(df.columns)

labelMapping = {"Normal/Mild": 0, "Moderate":1, "Severe":2}
dfMapped = df.replace(labelMapping)

def getLabelVectorWithoutSeg(studyId):
    try:
        labelVec = dfMapped.loc[studyId]
    except KeyError:
        labelVec = np.zeros_like(allLabels)

    return np.array(labelVec)

C:\Users\manue\AppData\Local\Temp\ipykernel_30560\1477235481.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfMapped = df.replace(labelMapping)


In [73]:

from tqdm import tqdm

mergedData = []
mergedLabels = []
for studId in tqdm(np.unique(studyIds)):
    idxs = np.argwhere(studyIds==studId).flatten()
    # print(np.array(instanceNrs)[idxs])
    # print(np.array(seriesIds)[idxs])
    # print(np.array(features)[idxs].shape)
    mergedData.append(np.mean(np.array(features)[idxs], axis=0))
    mergedLabels.append(getLabelVectorWithoutSeg(studId))

100%|██████████| 1975/1975 [01:53<00:00, 17.47it/s]


In [70]:
mergedData = np.array(mergedData)
mergedData.shape

(1975, 512)

In [75]:
X_train, X_test, y_train, y_test = train_test_split(np.array(mergedData), np.array(mergedLabels), test_size=0.33, random_state=42)


clfRF = RandomForestClassifier()
multi_target_rf = MultiOutputClassifier(clfRF, n_jobs=-1)

# Fit the model
multi_target_rf.fit(X_train, y_train)


# Predict on the test set
Y_pred = multi_target_rf.predict(X_test)

# Evaluate accuracy for each label separately
accuracies = []
f1s = []
for i in range(y_test.shape[1]):
    acc = accuracy_score(y_test[:, i], Y_pred[:, i])
    f1 = f1_score(y_test[:, i], Y_pred[:, i], average="micro")
    accuracies.append(acc)
    f1s.append(f1)
    print(f'Accuracy for label {i + 1}: {acc:.2f}')
    print(f'F1 for label {i + 1}: {f1:.2f}')

# Calculate and print the mean accuracy across all labels
mean_accuracy = np.mean(accuracies)
print(f'Mean accuracy across all labels: {mean_accuracy:.2f}')
mean_f1 = np.mean(f1s)
print(f'Mean F1 across all labels: {mean_f1:.2f}')


Accuracy for label 1: 0.95
F1 for label 1: 0.95
Accuracy for label 2: 0.90
F1 for label 2: 0.90
Accuracy for label 3: 0.84
F1 for label 3: 0.84
Accuracy for label 4: 0.76
F1 for label 4: 0.76
Accuracy for label 5: 0.96
F1 for label 5: 0.96
Accuracy for label 6: 0.96
F1 for label 6: 0.96
Accuracy for label 7: 0.90
F1 for label 7: 0.90
Accuracy for label 8: 0.75
F1 for label 8: 0.75
Accuracy for label 9: 0.60
F1 for label 9: 0.60
Accuracy for label 10: 0.61
F1 for label 10: 0.61
Accuracy for label 11: 0.96
F1 for label 11: 0.96
Accuracy for label 12: 0.90
F1 for label 12: 0.90
Accuracy for label 13: 0.75
F1 for label 13: 0.75
Accuracy for label 14: 0.60
F1 for label 14: 0.60
Accuracy for label 15: 0.63
F1 for label 15: 0.63
Accuracy for label 16: 0.92
F1 for label 16: 0.92
Accuracy for label 17: 0.83
F1 for label 17: 0.83
Accuracy for label 18: 0.67
F1 for label 18: 0.67
Accuracy for label 19: 0.48
F1 for label 19: 0.48
Accuracy for label 20: 0.72
F1 for label 20: 0.72
Accuracy for label